# 01 - Collect Play Store reviews (UPI apps)

**Question this project answers:** *Which complaint themes cost UPI apps the most rating points, and which fix is worth the most?*

**This notebook (Phase 1):** collects public Google Play Store reviews for five UPI apps and saves one raw CSV.

**How to run:** run each cell from top to bottom with **Shift+Enter**, and read the output before moving on.
Sections 1 to 3 are quick setup and checks. Section 4 does the real collection and takes a while, because the large apps have tens of thousands of reviews in the window.

**Privacy:** reviewer names and profile pictures are never saved. Only the review text, rating, dates and app version are kept.

## 1. Settings

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import shutil
import time

import pandas as pd
from google_play_scraper import app as app_info, reviews, Sort

# ---- Find the project root (the folder that contains requirements.txt) ----
def find_project_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Could not find requirements.txt. Open the project folder in VS Code first.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
PARTIAL_DIR = RAW_DIR / "_partial"      # temporary per-app files; deleted after a successful run
RAW_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

# ---- Which apps to collect (name -> Play Store package id) ----
APPS = {
    "PhonePe":    "com.phonepe.app",
    "Google Pay": "com.google.android.apps.nbu.paisa.user",
    "Paytm":      "net.one97.paytm",
    "CRED":       "com.dreamplug.androidapp",
    "BHIM":       "in.org.npci.upiapp",
}

# ---- Collection settings (safe to change) ----
LANG, COUNTRY = "en", "in"     # Indian Play Store, English interface
MAX_PER_APP = 80000             # stop after this many reviews per app
WINDOW_DAYS = 60               # ...or when reviews are older than this many days (the same window for every app)
CUTOFF = datetime.now() - timedelta(days=WINDOW_DAYS)
BATCH_SIZE = 200               # reviews fetched per request
PAUSE_SECONDS = 1.5            # polite pause between requests
FORCE_RECOLLECT = False        # True = ignore saved partial files and start over

print("Project root:", ROOT)
print("Raw data folder:", RAW_DIR)

Project root: C:\upi-reviews-analysis
Raw data folder: C:\upi-reviews-analysis\data\raw


## 2. Check the app IDs

Before the long run, confirm each app ID is valid. Every line should start with **OK**.
If one says **FAIL**, tell me which one and paste the message.

In [2]:
for name, pkg in APPS.items():
    try:
        info = app_info(pkg, lang=LANG, country=COUNTRY)
        score = info.get("score")
        ratings = info.get("ratings")
        score_txt = f"{score:.2f}" if score is not None else "n/a"
        ratings_txt = f"{ratings:,}" if ratings is not None else "n/a"
        print(f"OK   {name:<11} -> {info['title']} | avg rating {score_txt} | total ratings {ratings_txt}")
    except Exception as e:
        print(f"FAIL {name:<11} ({pkg}): {type(e).__name__}: {e}")

OK   PhonePe     -> PhonePe UPI Payments, Loan App | avg rating 4.43 | total ratings 14,329,834
OK   Google Pay  -> Google Pay: Secure UPI payment | avg rating 4.34 | total ratings 11,399,959
OK   Paytm       -> Paytm: Secure UPI Payments | avg rating 4.68 | total ratings 24,481,026
OK   CRED        -> CRED: Credit Cards, Bills, UPI | avg rating 4.76 | total ratings 2,998,671
OK   BHIM        -> BHIM Bharat's Own Payments App | avg rating 4.47 | total ratings 1,927,167


## 3. The collection function

This pulls reviews newest-first, a batch at a time, and stops at the review cap or the 12-month cutoff.
It retries on temporary errors, and it **never stores reviewer names**.

In [3]:
def collect_app(name, pkg):
    """Collect reviews for one app and return them as a DataFrame."""
    rows, token, batches = [], None, 0

    while len(rows) < MAX_PER_APP:
        # --- fetch one batch, retrying up to 3 times on errors ---
        batch = None
        for attempt in range(1, 4):
            try:
                batch, token = reviews(
                    pkg, lang=LANG, country=COUNTRY, sort=Sort.NEWEST,
                    count=BATCH_SIZE, continuation_token=token,
                )
                break
            except Exception as e:
                print(f"   retry {attempt}/3 after {type(e).__name__}")
                time.sleep(5 * attempt)
        if batch is None:
            print("   giving up after 3 failed tries; keeping what we have so far")
            break
        if not batch:
            break  # no more reviews available

        reached_cutoff = False
        for r in batch:
            if r["at"] < CUTOFF:
                reached_cutoff = True
                continue
            rows.append({
                "app": name,
                "review_id": r.get("reviewId"),
                "rating": r.get("score"),
                "text": r.get("content"),
                "thumbs_up": r.get("thumbsUpCount"),
                "review_version": r.get("reviewCreatedVersion"),
                "app_version": r.get("appVersion"),
                "review_date": r.get("at"),
                "has_dev_reply": r.get("replyContent") is not None,
            })
            # NOTE: userName and userImage are deliberately NOT kept (privacy).

        batches += 1
        if batches % 5 == 0:
            print(f"   {name}: {len(rows):,} reviews so far...")
        if reached_cutoff:
            break
        if token is None or getattr(token, "token", True) is None:
            break  # last page
        time.sleep(PAUSE_SECONDS)

    df = pd.DataFrame(rows).iloc[:MAX_PER_APP]
    df["collected_on"] = datetime.now().strftime("%Y-%m-%d")
    return df

## 4. Run the collection

This is the long step. Each app is saved to a temporary file as soon as it finishes, so if the run stops
(for example the network drops), run this cell again and it will **skip the apps already done**.

In [4]:
for name, pkg in APPS.items():
    slug = name.lower().replace(" ", "_")
    out = PARTIAL_DIR / f"{slug}.csv"

    if out.exists() and not FORCE_RECOLLECT:
        print(f"SKIP {name}: already collected ({out.name})")
        continue

    print(f"Collecting {name} ...")
    df_app = collect_app(name, pkg)
    df_app.to_csv(out, index=False)
    print(f"DONE {name}: {len(df_app):,} reviews saved\n")

   PhonePe: 1,000 reviews so far...
   PhonePe: 2,000 reviews so far...
   PhonePe: 3,000 reviews so far...
   PhonePe: 4,000 reviews so far...
   PhonePe: 5,000 reviews so far...
   PhonePe: 6,000 reviews so far...
   PhonePe: 7,000 reviews so far...
   PhonePe: 8,000 reviews so far...
   PhonePe: 9,000 reviews so far...
   PhonePe: 10,000 reviews so far...
   PhonePe: 11,000 reviews so far...
   PhonePe: 12,000 reviews so far...
   PhonePe: 13,000 reviews so far...
   PhonePe: 14,000 reviews so far...
   PhonePe: 15,000 reviews so far...
   PhonePe: 16,000 reviews so far...
   PhonePe: 17,000 reviews so far...
   PhonePe: 18,000 reviews so far...
   PhonePe: 19,000 reviews so far...
   PhonePe: 20,000 reviews so far...
   PhonePe: 21,000 reviews so far...
   PhonePe: 22,000 reviews so far...
   PhonePe: 23,000 reviews so far...
   PhonePe: 24,000 reviews so far...
   PhonePe: 25,000 reviews so far...
   PhonePe: 26,000 reviews so far...
   PhonePe: 27,000 reviews so far...
   PhonePe

## 5. Combine, summarise, and save the raw file

Creates `data/raw/reviews_raw.csv` and a short collection note. Then it deletes the temporary files.

In [5]:
files = sorted(PARTIAL_DIR.glob("*.csv"))
if len(files) < len(APPS):
    print(f"WARNING: only {len(files)} of {len(APPS)} apps collected. Re-run the cell above before combining.")

raw = pd.concat([pd.read_csv(f, parse_dates=["review_date"]) for f in files], ignore_index=True)
raw.to_csv(RAW_DIR / "reviews_raw.csv", index=False)

summary = raw.groupby("app").agg(
    reviews=("review_id", "count"),
    first_date=("review_date", "min"),
    last_date=("review_date", "max"),
    avg_rating=("rating", "mean"),
)
summary["avg_rating"] = summary["avg_rating"].round(2)
summary["days_covered"] = (summary["last_date"] - summary["first_date"]).dt.days

print("Total reviews:", f"{len(raw):,}")
display(summary)
print("Rating mix by app (% of reviews):")
display((pd.crosstab(raw["app"], raw["rating"], normalize="index") * 100).round(1))

Total reviews: 66,475


,reviews,first_date,last_date,avg_rating,days_covered
app,,,,,
BHIM,4257,2026-08-28 17:17:08,2026-10-26 17:10:05,2.77,58
CRED,1749,2026-08-28 18:59:08,2026-10-26 16:48:26,3.36,58
Google Pay,8559,2026-08-28 17:12:01,2026-10-26 17:15:01,4.08,59
Paytm,21951,2026-08-28 17:12:37,2026-10-26 17:15:31,4.48,59
PhonePe,29959,2026-08-28 17:14:04,2026-10-26 17:10:20,4.38,58


Rating mix by app (% of reviews):


rating,1,2,3,4,5
app,,,,,
BHIM,43.8,7.7,8.5,8.3,31.8
CRED,34.9,2.9,3.5,9.3,49.4
Google Pay,16.5,3.0,4.3,8.6,67.6
Paytm,9.8,1.3,1.9,5.0,82.0
PhonePe,9.4,1.8,3.7,11.3,73.7


In [ ]:
# Warn if any app covers too short a time window to study trends
for app_name, row in summary.iterrows():
    if row["days_covered"] < 30:
        print(f"NOTE: {app_name} covers only {row['days_covered']} days. "
              f"Trend analysis will be thin for it. We can raise MAX_PER_APP if needed.")

# Write a plain-text collection note (goes into the repo as documentation)
note = f"""# Data collection note

- Source: Google Play Store public reviews (country={COUNTRY}, language={LANG})
- Collected on: {datetime.now().strftime('%Y-%m-%d')}
- Apps: {', '.join(APPS)}
- Window: last {WINDOW_DAYS} days ({raw['review_date'].min():%Y-%m-%d} to {raw['review_date'].max():%Y-%m-%d})
- Method: google-play-scraper library, newest first, up to {MAX_PER_APP:,} reviews per app or the last {WINDOW_DAYS} days, whichever comes first
- Total reviews: {len(raw):,}
- Privacy: reviewer names and profile pictures were not stored
- Limitation: a snapshot of public reviews; reviews skew toward extreme opinions and do not represent all users

```
{summary.to_string()}
```
"""
(RAW_DIR / "collection_note.md").write_text(note, encoding="utf-8")

# Clean up temporary files only if every app was collected
if len(files) == len(APPS):
    shutil.rmtree(PARTIAL_DIR)
    print("Saved: data/raw/reviews_raw.csv and data/raw/collection_note.md. Temporary files removed.")
else:
    print("Temporary files kept because collection is incomplete.")

Saved: data/raw/reviews_raw.csv and data/raw/collection_note.md. Temporary files removed.


## Next

- Look at the summary table above. Each app should have a few thousand reviews.
- Commit and push `data/raw/` so the raw snapshot is backed up on GitHub.
- **Phase 2:** cleaning and profiling (`02_clean.ipynb`) is next.